#Load Libraries

In [1]:
# Import necessary libraries
import os                     # OS module provides a portable way of using operating system dependent functionality.
import numpy as np            # NumPy is a library used for working with arrays. It also has functions for working in domain of linear algebra, 
                              # fourier transform, and matrices.
import cv2                    # OpenCV (Open Source Computer Vision) is a library used for computer vision applications.
from sklearn import preprocessing    # Preprocessing module provides several functions for scaling, transforming and wrangling data. 
                                      # It is used in machine learning and data science.
from sklearn.model_selection import train_test_split   # Sklearn.model_selection module contains functions for splitting data into training 
                                                       # and testing sets.
from keras.utils import to_categorical   # The to_categorical() function is used to convert a class vector (integers) to binary class matrix.
import matplotlib.pyplot as plt         # Matplotlib is a library used for plotting graphs and visualizations.


In [2]:
#Importing the "drive" module from the "google.colab" library
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/LUMBAR_SPINE/

/content/drive/MyDrive/LUMBAR_SPINE


# Download Data

## Download

In [4]:
#!wget https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/zbf6b4pttk-2.zip

## Unzip

In [5]:
#!unzip zbf6b4pttk-2.zip

In [6]:
#!unzip 'Label Image Ground Truth Data for Lumbar Spine MRI Dataset/Ground_Truth_Label.zip'

In [7]:
#!unzip 'Label Image Ground Truth Data for Lumbar Spine MRI Dataset/Manual_Label_Data.zip'

# Peprocess and Save Data

In [8]:
proj_dir='/content/drive/MyDrive/LUMBAR_SPINE/'

In [9]:
images_path=proj_dir+'05_Final_Ground_Truth_Data/Composite_Images/'
labels_path=proj_dir+'05_Final_Ground_Truth_Data/Label_Images/'

In [10]:
###load image masks into a array###
train_masks=[]
for img_path in os.listdir(labels_path):
  img=cv2.imread(labels_path+img_path,0)
  img=cv2.resize(img, (320,320), interpolation = cv2.INTER_AREA)
  img=(img==50)*1
  train_masks.append(img)
train_masks=np.array(train_masks)

In [11]:
###load images into a array###
train_imgs=[]
for img_path in os.listdir(images_path):
  img=cv2.imread(images_path+img_path)
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
  img=cv2.resize(gray, (320,320), interpolation = cv2.INTER_AREA)
  train_imgs.append(img)
train_imgs=np.array(train_imgs)

In [12]:
train_masks.shape

(1545, 320, 320)

In [13]:
train_imgs.shape

(1545, 320, 320)

In [14]:
##convert labels into categorical##
le=preprocessing.LabelEncoder()
n,h,w=train_masks.shape
train_masks_reshape=train_masks.reshape(-1,1)
train_masks_reshape_en=le.fit_transform(train_masks_reshape)
train_masks_en=train_masks_reshape_en.reshape(n,h,w)
train_masks_cat=to_categorical(train_masks_en,2)

/usr/local/lib/python3.9/dist-packages/sklearn/preprocessing/_label.py:116: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [15]:
np.unique(train_masks_en)

array([0, 1])

In [16]:
##split data into train and test sets##
X_train, X_test,train_masks_cat, test_masks_cat = train_test_split(train_imgs, train_masks_cat, test_size=0.10, random_state=42)

In [17]:
train_masks_cat.shape

(1390, 320, 320, 2)

In [18]:
##save data##
np.save(proj_dir+'X_train.npy', X_train)
np.save(proj_dir+'train_masks_cat.npy', train_masks_cat)
np.save(proj_dir+'X_test.npy', X_test)
np.save(proj_dir+'test_masks_cat.npy', test_masks_cat)